In [ ]:
#%matplotlib widget
import json
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import io, h5py, zipfile
import numpy as np
from tqdm import tqdm
import os, sys
sys.path.append(os.path.abspath(".."))
import mienc.support as ms
from mienc import Corrector, NonLinearEstimator
import pandas as pd
from scipy.signal import hilbert
from scipy.io import loadmat, savemat
from scipy.stats import ks_1samp, uniform, pearsonr, spearmanr
import seaborn as sns

In [ ]:
mat = loadmat("/mnt/DATA/Motion_fMRI/Datasets/MOVZ/motion_movz.mat")
print(mat.keys())

In [ ]:
print(mat['DVARSmean'].shape, mat['FDmean'].shape)
mat['DVARSmean'][0,0].shape, mat['FDmean'][0,0].shape

In [ ]:
wtf = np.load("../../NonLinearityResultsNew/localised/fMRI_AAL90_region_quantiles_strin.npy")
wtf.shape

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

def cmap_from_values (values, base_cmap, vmin=None, vmax=None, boundaries=None):
    if vmin is None:
        vmin = np.min(values)
    if vmax is None:
        vmax = np.max(values)
    if boundaries is None:
        boundaries = 1 - np.linspace(0,1,len(values),False)
    cdict = {
        'red': [],
        'green': [],
        'blue': [],
        'alpha': []
    }
    tmpdict = {
        'red': [0, 0],
        'green': [0, 0],
        'blue': [0, 0],
        'alpha': [0, 0]
    }
    for i in range(90):
        tv = (values[i]-vmin)/(vmax-vmin)
        tc = base_cmap(tv)
        for j, channel in enumerate(tmpdict):
            tmpdict[channel].append(tc[j])
            cdict[channel].append(tuple(tmpdict[channel]))
            tmpdict[channel] = [boundaries[i], tc[j]]
    for j, channel in enumerate(tmpdict):
        tmpdict[channel].append(0)
        cdict[channel].append(tuple(tmpdict[channel]))

    return LinearSegmentedColormap("real", cdict)


In [ ]:
from nilearn import datasets, plotting, image
import nibabel as nib
import pandas as pd
aal_atlas_centers = pd.read_csv("../auxiliary_data/AAL_90regions.csv")
aal_atlas_centers_labels=aal_atlas_centers["Labels"].apply(lambda x: x.split()[1]).tolist()
atlas_data = datasets.fetch_atlas_aal()
atlas_filename = atlas_data.maps
region_indices = {l:i for l,i in zip(atlas_data["labels"],atlas_data["indices"])}
good = np.array([int(region_indices[l]) for l in region_indices if l in aal_atlas_centers_labels])
steps_norm = (good-good[0])/(good[-1]-good[0])
boundaries = np.concatenate([steps_norm[:-1]+np.diff(steps_norm)/2, [1]])

base_palette = plt.cm.magma
base_palette.set_under(base_palette.get_bad())
base_palette.set_over(base_palette.get_bad())
base_palette.set_bad(color="grey")
cmaplist = [base_palette(i) for i in np.arange(11)*25]#np.concatenate([np.arange(6)*25, 151+np.arange(5)*26])
cool_palette = LinearSegmentedColormap.from_list('Custom cmap', cmaplist, 6)
cool_palette.set_under(cool_palette.get_bad())
shad = np.load(f"fMRI_AAL90_sha_KS_nonLin.npy")
real = np.load(f"fMRI_AAL90_KS_nonLin.npy")
max_ks = np.nanmax([real, shad])
min_ks = np.nanmin([real, shad])

def plot_brain_old(cmap, title=None, cut_coords=None, axes=None):
    plotting.plot_roi(atlas_filename, title=title, cut_coords=cut_coords, cmap=cmap, vmin=2001, vmax=8302, axes=axes)
    plt.show()

In [ ]:
cool_palette

plotting functions

In [ ]:
from nilearn import datasets, plotting, image
import nibabel as nib
import pandas as pd
from scipy.spatial import Voronoi, voronoi_plot_2d
aal_atlas_centers = pd.read_csv("../auxiliary_data/AAL_90regions.csv")
aal_atlas_centers_labels=aal_atlas_centers["Labels"].apply(lambda x: x.split()[1]).tolist()
atlas_data = datasets.fetch_atlas_aal()
atlas_filename = atlas_data.maps
region_indices = {l:i for l,i in zip(atlas_data["labels"],atlas_data["indices"])}

reg_loc = []
image_data = image.get_data(atlas_filename)
affine=image.load_img(atlas_filename).affine
for l in region_indices:
    if l in aal_atlas_centers_labels:
        reg_loc.append(np.where(image_data==int(region_indices[l])))
        
def plot_brain(in_array, title=None, cut_coords=None, axes=None, cmap=base_palette, vmin=None, vmax=None):
    MAXINT = np.iinfo(np.int32).max
    if vmin is None:
        vmin = np.nanmin(in_array)
    if vmax is None:
        vmax = np.nanmax(in_array)
    FACTOR = MAXINT / (max(np.abs(vmin), np.abs(vmax))+1)
    values = np.full_like(image_data,np.nan, dtype=np.int32)
    for i in range(90):
        values[reg_loc[i][0],reg_loc[i][1],reg_loc[i][2]]=in_array[i]*FACTOR

    img = nib.nifti1.Nifti1Image(values, affine)
    print(FACTOR, vmin*FACTOR, vmax*FACTOR, np.nanmin(in_array[in_array>0]), np.nanmax(in_array), np.nanmin(values[values>0]), np.nanmax(values))
    plotting.plot_roi(img, title=title, cut_coords=cut_coords, cmap=cmap, vmin=vmin*FACTOR, vmax=vmax*FACTOR, axes=axes)

def plot_cap(in_array, title=None, region_positions=None, axes=None, cmap=base_palette, vmin=None, vmax=None, plane_distance=10):
    source_z = region_positions.z.min()
    diameter = region_positions.z.max()-source_z
    region_positions["XP"] = region_positions.x/(region_positions.z-source_z+plane_distance)*(diameter+plane_distance)
    region_positions["YP"] = region_positions.y/(region_positions.z-source_z+plane_distance)*(diameter+plane_distance)
    if vmin is None:
        vmin = np.nanmin(in_array)
    if vmax is None:
        vmax = np.nanmax(in_array)
    vor = Voronoi(np.concatenate([region_positions[["XP","YP"]],np.transpose([22*np.sin(np.linspace(0,2*np.pi,100)),22*np.cos(np.linspace(0,2*np.pi,100))-0.5])]))
    normalised = (in_array-vmin)/(vmax-vmin)
    if axes is not None:
        plt.sca(axes)
    plt.gca().set_aspect('equal')

    for i, v in enumerate(normalised):
        region = vor.regions[vor.point_region[i]]
        vertices =np.array([vor.vertices[n] for n in region])
        plt.fill(vertices[:,0], vertices[:,1], color=cmap(v))
    if title is not None:
        plt.title(title)
    # voronoi_plot_2d(vor, show_vertices=False, point_size=4)
    # plt.plot(37*np.sin(np.linspace(0,2*np.pi,50)),37*np.cos(np.linspace(0,2*np.pi,50))-5, "-r")

def HolmThresholdFromP (p_values:np.ndarray):
    """Returns the Holm-Bonferroni threshold given an array of p-values.
    NOTA BENE: reject the null hypotheses when **strictly smaller** than this thresold. This corresponds to the *p-value* of the first non-rejected null hypothesis.

    Parameters
    ----------
    p_values : np.ndarray
        The *p-values* to consider, can be N-dimensional, will be flattened.

    Returns
    -------
    float
        The *p-value* of the first non-rejected hypothesis.
    """
    sorted_p = np.sort(p_values.flatten())
    good = sorted_p < 0.05/(sorted_p.size-np.arange(sorted_p.size))
    which = np.argmin(good)
    
    if which==0 and sorted_p[-1]<0.05:
        return 0.05
    
    return sorted_p[which]

In [ ]:
samp = np.random.randint(100, size=242)-2
samp[samp>99]=99
samp[samp<0]=0
ks_1samp(samp, uniform.cdf,(0,100), alternative="less")

In [ ]:
def is_significance_correlated (results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps):
    global MAX_CM, MAX_CM2
    corrct = Corrector(nbins, nsteps, folder_name=os.path.dirname(results_file).format(subset_identifiers[0]), config="../config.ini", cache_dir="../cache/")
    corrct.compute_correction()
    for subset_id, subset_na in zip(subset_identifiers, subset_names):
        region_quantiles=np.zeros((subj_num, pair_num))
        if os.path.isfile(f"../../NonLinearityResultsNew/localised/{output_prefix}_region_quantiles_{subset_id}.npy"):
            region_quantiles = np.load(f"../../NonLinearityResultsNew/localised/{output_prefix}_region_quantiles_{subset_id}.npy")
            print(f"Region quantiles for {subset_description} {subset_id} loaded")
        else:
            for s in tqdm(range(subj_num), "Subject"):
                pat = np.load(results_file.format(subset_id, s))
                true = pat[:,0]
                surr = np.sort(pat[:,1:], 1)
                for i in range(pair_num):
                    region_quantiles[s,i] = np.searchsorted(surr[i,:], true[i], "left")
                np.save(f"../../NonLinearityResultsNew/localised/{output_prefix}_region_quantiles_{subset_id}.npy", region_quantiles)
        
        fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [2, 3]}, figsize=(8,5))
        plt.sca(ax[0])
        plt.hist(region_quantiles[:,0], bins=np.arange(0,110,10))
        plt.ylabel("# subjects")
        plt.xlabel("%ile")

        plt.sca(ax[0].twinx())
        who, counts = np.unique(region_quantiles[:,0]+1, return_counts=True)
        truecum = np.cumsum(counts)/subj_num
        exp = np.arange(0,101)/100
        plt.plot(who, truecum, color="black")
        plt.plot(exp, color="orange")
        diff = exp[who.astype(int)]-truecum
        ks_stat = np.max(diff)
        where = np.argmax(diff)
        plt.vlines(who[where], truecum[where], exp[int(who[where])],"red")
        plt.title(f"K-S statistic: {ks_stat:.3} at {who[where]}")
        plt.xlim((0,100))
        plt.ylim((0,1))
        plt.ylabel("cumulative")
        plt.title(subset_description+subset_na+" - Sample K-S plot")

        ks_stat = np.zeros(subj_num)
        ks_p = np.zeros(subj_num)
        exp = np.arange(0,101)/100
        for s in tqdm(range(subj_num), "Subj"):
            stat, pval = ks_1samp(region_quantiles[s, :], uniform.cdf,(0,100), alternative="less")
            ks_stat[s] = stat
            ks_p[s] = pval

        plt.sca(ax[1])
        sorted_p = np.sort(ks_p.flatten())
        plt.plot(sorted_p)
        plt.plot(np.arange(subj_num),0.05/(subj_num-np.arange(subj_num)),":r")
        plt.plot([0,subj_num-1],[0.05/subj_num,0.05/subj_num], ":k")
        plt.yscale("log")

        thresh = HolmThresholdFromP(sorted_p)
        ax[1].yaxis.tick_right()
        ax[1].yaxis.set_label_position("right")
        plt.ylabel("$p$-value")
        plt.xlabel("subject")
        plt.title(f"Holm-Bonferroni threshold: {thresh:.3} -> {np.min(ks_stat[ks_p<thresh]):.3}")
        plt.show()
    return ks_stat
        


In [ ]:
pair_num = 4005
subj_num = 245
elec_num = 90
quantiles = [.6,.9]
results_file = "../../NonLinearityResultsNew/eso245_aal_{}_90_ExtSha_bin8/subject{:02}_8.npy"#"../../NonLinearityResults/eso245_aal_{}_90_bin9/patient{:02}_9.npy"
subset_description = "fMRI AAL90 "
subset_identifiers = ["strin"]# ["raw", "mod", "strin"]
subset_names = ["strin"]# ["raw", "mod", "strin"]
output_prefix = "fMRI_AAL90"
region_positions = pd.read_csv("../auxiliary_data/AAL_90regions.csv").loc[slice(None), ["Labels","x","y","z"]]
region_positions["Labels"] = region_positions["Labels"].apply(lambda x: x.split()[1])
nbins = 9
nsteps = loadmat(f"/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_aal_strin_90.mat")["subj_tcs"].shape[0]
ks_stat_true = is_significance_correlated(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps)
subset_description = "SHA fMRI AAL90 "
output_prefix = "fMRI_AAL90_sha"
results_file = "../../NonLinearityResultsNew/eso245_aal_{}_90_ExtSha_bin8/subject{:02}_8_sha.npy"#"../../NonLinearityResults/eso245_aal_{}_90_bin9/patient{:02}_9_sha.npy"
ks_stat_shad = is_significance_correlated(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps)
pearsonr(ks_stat_true, ks_stat_shad)

In [ ]:
plt.scatter(ks_stat_true, ks_stat_shad)
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.vlines(0.03332084, *plt.ylim(), "k", "dotted")
plt.hlines(0.03059925, *plt.xlim(), "k", "dotted")
plt.xlabel("True")
plt.ylabel("Shadow")
plt.show()

main function

In [ ]:
MAX_CM = 0
MAX_CM2 = 0

def summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps, cut_position):
    global MAX_CM, MAX_CM2
    corrct = Corrector(nbins, nsteps, folder_name=os.path.dirname(results_file).format(subset_identifiers[0]), config="../config.ini", cache_dir="../cache/")
    corrct.compute_correction()
    for subset_id, subset_na in zip(subset_identifiers, subset_names):
        region_quantiles=np.zeros((subj_num, pair_num))
        if os.path.isfile(f"../../NonLinearityResultsNew/localised/{output_prefix}_region_quantiles_{subset_id}.npy"):
            region_quantiles = np.load(f"../../NonLinearityResultsNew/localised/{output_prefix}_region_quantiles_{subset_id}.npy")
            print(f"Region quantiles for {subset_description} {subset_id} loaded")
        else:
            for s in tqdm(range(subj_num), "Subject"):
                pat = np.load(results_file.format(subset_id, s))
                true = pat[:,0]
                surr = np.sort(pat[:,1:], 1)
                for i in range(pair_num):
                    region_quantiles[s,i] = np.searchsorted(surr[i,:], true[i], "left")
                np.save(f"../../NonLinearityResultsNew/localised/{output_prefix}_region_quantiles_{subset_id}.npy", region_quantiles)
        

        who, counts = np.unique(region_quantiles[:,0]+1, return_counts=True)
        truecum = np.cumsum(counts)/subj_num
        exp = np.arange(0,101)/100
        diff = exp[who.astype(int)]-truecum
        ks_stat = np.max(diff)
        where = np.argmax(diff)

        fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [2, 3]}, figsize=(8,5))
        plt.sca(ax[0])
        plt.hist(region_quantiles[:,0], bins=np.arange(0,110,10))
        plt.ylabel("# pairs")
        plt.xlabel("%ile")

        plt.sca(ax[0].twinx())
        plt.plot(who, truecum, color="black")
        plt.plot(exp, color="orange")
        plt.vlines(who[where], truecum[where], exp[int(who[where])],"red")
        plt.title(f"K-S statistic: {ks_stat:.3} at {who[where]}")
        plt.xlim((0,100))
        plt.ylim((0,1))
        plt.ylabel("cumulative")
        plt.title(subset_description+subset_na+" - Sample K-S plot")

        if os.path.isfile(f"../../NonLinearityResultsNew/localised/{output_prefix}_ks_stat_{subset_id}.npy") and os.path.isfile(f"../../NonLinearityResultsNew/localised/{output_prefix}_ks_p_{subset_id}.npy"):
            ks_stat = np.load(f"../../NonLinearityResultsNew/localised/{output_prefix}_ks_stat_{subset_id}.npy")
            ks_p = np.load(f"../../NonLinearityResultsNew/localised/{output_prefix}_ks_p_{subset_id}.npy")
            print(f"K-S stats for {subset_description} {subset_id} loaded")
        else:
            ks_stat = np.zeros(pair_num)
            ks_p = np.zeros(pair_num)
            exp = np.arange(0,101)/100
            for i in tqdm(range(pair_num), "Pair"):
                stat, pval = ks_1samp(region_quantiles[:,i], uniform.cdf,(0,100), alternative="less")
                ks_stat[i] = stat
                ks_p[i] = pval
            np.save(f"../../NonLinearityResultsNew/localised/{output_prefix}_ks_stat_{subset_id}.npy", ks_stat)
            np.save(f"../../NonLinearityResultsNew/localised/{output_prefix}_ks_p_{subset_id}.npy", ks_p)

        thresh = HolmThresholdFromP(ks_p)
        print("HB thresh:", thresh)
        plt.sca(ax[1])
        plt.plot(np.sort(ks_p.flatten()))
        plt.plot(np.arange(pair_num),0.05/(pair_num-np.arange(pair_num)),":r")
        plt.plot([0,pair_num-1],[0.05/pair_num,0.05/pair_num], ":k")
        plt.yscale("log")

        ax[1].yaxis.tick_right()
        ax[1].yaxis.set_label_position("right")
        plt.ylabel("$p$-value")
        plt.xlabel("pair")
        if (ks_p<thresh).any():
            plt.title(f"Holm-Bonferroni threshold: {thresh:.3} -> {np.min(ks_stat[ks_p<thresh]):.3}")
        else:
            plt.title(f"Holm-Bonferroni threshold: {thresh:.3} -> --")
        plt.show()
        
        corrected = np.full(pair_num, np.nan)
        corrected[ks_p<thresh] = ks_stat[ks_p<thresh]
        mappa = np.zeros([elec_num,elec_num])
        mappa[np.triu_indices(elec_num,1)]=corrected
        mappa+=mappa.T
        siginreg = np.sum(mappa>0, 1)
        print("Non linear connections:", np.sum(siginreg)/2)

        fix, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [1.5,2]}, figsize=(12,5))
        plt.sca(ax[0])
        plt.imshow(mappa[siginreg>0,:][:,siginreg>0], interpolation="none")
        plt.colorbar(shrink=0.7).ax.set_ylabel('KS statistcs', rotation=90)
        plt.yticks(np.arange(0,sum(siginreg>0),10), np.arange(elec_num)[siginreg>0][::10])
        step = 10 if len(np.arange(0,sum(siginreg>0),10))<12 else 20
        plt.xticks(np.arange(0,sum(siginreg>0),step), np.arange(elec_num)[siginreg>0][::step])
        plt.xlabel(f"{sum(siginreg>0)}/{elec_num} electrodes")

        plt.sca(ax[1])
        plt.plot(np.sum(mappa>0, 1), "o")
        plt.xticks(np.arange(0,elec_num,10), rotation=45)
        plt.xlabel(f"Electrode")
        plt.ylabel("# significant non-linear connections")
        plt.suptitle(subset_description+subset_na+f" - {np.sum(ks_p<thresh)} ({100*np.sum(ks_p<thresh)/pair_num:.3} %) significant pairs")
        plt.show()

        if region_positions is not None and (siginreg>0).any():
            fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [8,1]},figsize=(8,6))
            ax1 = fig.add_subplot(121,projection='3d')
            ax[0].axis("off")
            # linear = region_positions[siginreg==0]
            lt, mt = np.quantile(siginreg[siginreg>0], quantiles)
            other_code = region_positions
            # low = region_positions[np.logical_and(siginreg>0, siginreg<=lt)]
            # mild = region_positions[np.logical_and(siginreg>lt, siginreg<=mt)]
            # very = region_positions[siginreg>mt]
            palette = plt.cm.magma
            palette.set_under(color="grey")
            MAX_CM = np.max(siginreg)#max(MAX_CM, np.max(siginreg))
            # ax1.scatter(linear.x, linear.y, linear.z, marker="o", color="gray")
            mpb = ax1.scatter(other_code.x, other_code.y, other_code.z, marker="o", c=siginreg, cmap=palette, vmin=1, vmax=MAX_CM)
            # ax1.scatter(low.x, low.y, low.z, marker="o", color="yellow")
            # ax1.scatter(mild.x, mild.y, mild.z, marker="o", color="orange")
            # ax1.scatter(very.x, very.y, very.z, marker="o", color="red")

            ax1.set_xlabel('Frontal')
            ax1.set_ylabel('Anteroposterior')
            ax1.set_zlabel('Craniocaudal')

            plt.colorbar(mpb, cax=ax[1], shrink=0.6, extend="min")
            plt.sca(ax[1])
            plt.ylabel('# non-linear connections', rotation=90)
            # legend_elements = [Line2D([0], [0], marker='o', color='w', label='0', markerfacecolor='gray', markersize=15),
            #                 Line2D([0], [0], marker='o', color='w', label=f'1 - {lt:.3} ({int(quantiles[0]*100)} %ile)', markerfacecolor='yellow', markersize=15),
            #                 Line2D([0], [0], marker='o', color='w', label=f'{lt:.3} - {mt:.3} ({int(quantiles[1]*100)} %ile)', markerfacecolor='orange', markersize=15),
            #                 Line2D([0], [0], marker='o', color='w', label=f'> {mt:.3}', markerfacecolor='red', markersize=15)]
            # ax[1].legend(handles=legend_elements, loc='center', title="Number of significantly\nnon linear connections", frameon=False)
            # ax[1].axis("off")
            plt.suptitle(subset_description+subset_na+" - Non-Linearity position")
            plt.show()
            print("Siginreg max:", np.max(siginreg))
            fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [8,1]},figsize=(8,6))
            if "aal" in results_file:
                sc = plt.scatter([np.nan],[np.nan],c=0,cmap=palette,vmin=1, vmax=MAX_CM) #cool_
                cbar = plt.colorbar(sc, cax=ax[1], shrink=0.4)
                cbar.set_ticks((np.arange(MAX_CM)+0.5)*(MAX_CM-1)/MAX_CM+1)
                cbar.set_ticklabels([str(i) for i in range(1,MAX_CM+1)])
                cbar.set_label("# significantly non linear relationships")
                plot_brain(siginreg+0.5,"AAL 90 regions"+(" surrogate" if "sha" in output_prefix else ""),cut_position, ax[0], cool_palette, 1, MAX_CM)# (-15,-75,27)
            else:
                vmax=max(np.max(siginreg), 10)
                sc = plt.scatter([np.nan],[np.nan],c=0,cmap=palette,vmin=1, vmax=vmax)
                cbar = plt.colorbar(sc, cax=ax[1], shrink=0.4, extend="min")
                cbar.set_label("# significantly non linear relationships")
                plot_cap(siginreg, output_prefix+" - "+subset_description+subset_na+" - Non-Linearity position", region_positions, ax[0], palette, 1,vmax)
            plt.savefig(f"{output_prefix}_brain.pdf", bbox_inches="tight")
            plt.show()

        if os.path.isfile(f"../../NonLinearityResultsNew/localised/{output_prefix}_all_regions_quantiles_{subset_id}.npy") and os.path.isfile(f"../../NonLinearityResultsNew/localised/{output_prefix}_all_regions_MI_{subset_id}.npy"):
            all_regions_quantiles = np.load(f"../../NonLinearityResultsNew/localised/{output_prefix}_all_regions_quantiles_{subset_id}.npy")
            all_regions_MI = np.load(f"../../NonLinearityResultsNew/localised/{output_prefix}_all_regions_MI_{subset_id}.npy")
            print(f"All Regions quantiles for {subset_description} {subset_id} loaded")
        else:
            all_of_it = np.zeros((elec_num, elec_num, subj_num, 100))
            for i in tqdm(range(subj_num), desc="Load and correct"):
                all_of_it[(*np.triu_indices(elec_num,1),i,slice(None))] = corrct.correct(np.load(results_file.format(subset_id, i)))
            all_of_it+=all_of_it.transpose((1,0,2,3))
            all_regions = np.sum(all_of_it, axis=0)/(elec_num-1)

            all_regions_quantiles = np.zeros((subj_num, elec_num))
            for s in tqdm(range(subj_num), desc="Quantiles subj"):
                true = all_regions[:,s,0]
                surr = np.sort(all_regions[:,s,1:], 1)
                for i in range(elec_num):
                    all_regions_quantiles[s,i] = np.searchsorted(surr[i,:], true[i], "left")
            np.save(f"../../NonLinearityResultsNew/localised/{output_prefix}_all_regions_quantiles_{subset_id}.npy", all_regions_quantiles)
            all_regions_MI = np.zeros((elec_num, subj_num),[("TMI","f8"),("GMI","f8"),("sGMI","f8")])
            all_regions_MI["TMI"]=all_regions[:,:,0]
            all_regions_MI["GMI"]=np.mean(all_regions[:,:,1:], axis=-1)
            all_regions_MI["sGMI"]=np.std(all_regions[:,:,1:], axis=-1)
            np.save(f"../../NonLinearityResultsNew/localised/{output_prefix}_all_regions_MI_{subset_id}.npy", all_regions_MI)

        fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [1,1]},figsize=(10,6))
        rnl = 1 - all_regions_MI["GMI"]/all_regions_MI["TMI"]#(all_regions_MI["TMI"]-all_regions_MI["GMI"])/all_regions_MI["sGMI"]#
        num_reg_shown = 10
        sa = np.argsort(corrected)[::-1]
        to_show = np.argsort(siginreg)[-1:-11:-1]#np.roll(sa,-np.count_nonzero(np.isnan(corrected)))[:num_reg_shown]
        plt.sca(ax[0])
        plt.boxplot([rnl[i,:] for i in to_show], showmeans=True)
        plt.xlabel(f"{num_reg_shown} regions with most non-linear connections")
        plt.ylim(plt.ylim())
        # plt.vlines([num_reg_shown+0.5, num_reg_shown+2.5], *plt.ylim(), colors = "green", linestyles="dashed", lw=1)
        plt.ylabel("Relative non linearity")
        if region_positions is not None:
            labels = region_positions.loc[to_show].Labels.to_numpy()
        else:
            labels = np.arange(num_reg_shown+4)+1
        plt.xticks(np.arange(num_reg_shown)+1, labels, rotation=90)
        plt.xlim(plt.xlim())
        plt.hlines([0,0.1],*plt.xlim(), ["k","r"],["solid","dashed"])

        plt.sca(ax[1])
        top_reg = np.argmax(siginreg)
        top_sub = np.argsort(rnl[top_reg,:])[-15:]
        plt.barh(np.arange(15),all_regions_MI["TMI"][top_reg,top_sub],color="darkseagreen", label="Total MI")
        plt.barh(np.arange(15),all_regions_MI["GMI"][top_reg,top_sub], xerr=all_regions_MI["sGMI"][top_reg,top_sub],color="indianred", label="Surrogates MI")
        plt.ylabel(f"15 most nonlinear subjects for most \"nonlinear\" region ({region_positions.loc[to_show[0],'Labels'] if region_positions is not None else '-'})")
        plt.xlabel("MI")
        plt.legend(frameon=False,loc=0)
        plt.show()

        fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [2, 3]}, figsize=(8,5))
        plt.sca(ax[0])
        plt.hist(all_regions_quantiles[:,0], bins=np.arange(0,110,10))
        plt.ylabel("# pairs")
        plt.xlabel("%ile")

        plt.sca(ax[0].twinx())
        who, counts = np.unique(all_regions_quantiles[:,0]+1, return_counts=True)
        truecum = np.cumsum(counts)/subj_num
        exp = np.arange(0,101)/100
        plt.plot(who, truecum, color="black")
        plt.plot(exp, color="orange")
        diff = exp[who.astype(int)]-truecum
        ks_stat = np.max(diff)
        where = np.argmax(diff)
        plt.vlines(who[where], truecum[where], exp[int(who[where])],"red")
        plt.title(f"K-S statistic: {ks_stat:.3} at {who[where]}")
        plt.xlim((0,100))
        plt.ylim((0,1))
        plt.ylabel("cumulative")
        plt.title(subset_description+subset_na+" - Sample K-S plot - regions")
        
        regions_ks_stat = np.zeros(elec_num)
        regions_ks_p = np.zeros(elec_num)
        exp = np.arange(0,101)/100
        for i in range(elec_num):
            stat, pval = ks_1samp(all_regions_quantiles[:,i], uniform.cdf,(0,100), alternative="less")
            regions_ks_stat[i] = stat
            regions_ks_p[i] = pval

        corrected = np.full(elec_num, np.nan)
        thresh = HolmThresholdFromP(regions_ks_p)
        corrected[regions_ks_p<thresh] = regions_ks_stat[regions_ks_p<thresh]


        plt.sca(ax[1])
        plt.plot(np.sort(regions_ks_p.flatten()))
        plt.plot(np.arange(elec_num),0.05/(elec_num-np.arange(elec_num)),":r")
        plt.plot([0,elec_num-1],[0.05/elec_num,0.05/elec_num], ":k")
        plt.yscale("log")
        ax[1].yaxis.tick_right()
        ax[1].yaxis.set_label_position("right")
        plt.ylabel("$p$-value")
        plt.xlabel("pair")
        print(thresh, np.max(regions_ks_p), regions_ks_stat[regions_ks_p<thresh].shape)
        if (regions_ks_p<thresh).any():
            plt.title(f"Holm-Bonferroni threshold: {thresh:.3} -> {np.min(regions_ks_stat[regions_ks_p<thresh]):.3} - region")
        else:
            plt.title(f"Holm-Bonferroni threshold: {thresh:.3} -> --")
        plt.show()

        if region_positions is not None and np.isfinite(corrected).any():
            fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [8,1]},figsize=(8,6))
            ax1 = fig.add_subplot(121,projection='3d')
            ax[0].axis("off")
            palette = plt.cm.magma
            palette.set_under(color="grey")
            MAX_CM2 = np.nanmax(regions_ks_stat)#max(MAX_CM2, np.max(regions_ks_stat))
            # np.save(f"{output_prefix}_KS_nonLin", corrected)
            # ax1.scatter(region_positions[np.isnan(corrected)].x, region_positions[np.isnan(corrected)].y, region_positions[np.isnan(corrected)].z, marker="o", c="grey")
            scat = ax1.scatter(region_positions.x, region_positions.y, region_positions.z, marker="o", c=regions_ks_stat, cmap=palette, vmin=np.nanmin(corrected), vmax=MAX_CM2)
            plt.colorbar(scat, cax=ax[1], shrink=0.6, extend="min").ax.set_ylabel('KS statistcs', rotation=90)

            ax1.set_xlabel('Frontal')
            ax1.set_ylabel('Anteroposterior')
            ax1.set_zlabel('Craniocaudal')
            plt.suptitle("Non-Linearity position")
            plt.show()


            fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [8,1]},figsize=(8,6))
            if "aal" in results_file:
                sc = plt.scatter([np.nan],[np.nan],c=0,cmap=base_palette,vmin=min_ks, vmax=max_ks)
                plt.colorbar(sc, cax=ax[1], shrink=0.4)
                plot_brain(regions_ks_stat, output_prefix,cut_position, ax[0], cool_palette, min_ks, max_ks)
            else:
                sc = plt.scatter([np.nan],[np.nan],c=0,cmap=palette,vmin=np.nanmin(corrected), vmax=np.max(regions_ks_stat))
                cbar = plt.colorbar(sc, cax=ax[1], shrink=0.4, extend="min")
                cbar.set_label("KS statistcs")
                plot_cap(regions_ks_stat, output_prefix+" - "+subset_description+subset_na+" - Non-Linearity position", region_positions, ax[0], palette, np.nanmin(corrected), np.nanmax(corrected))
            plt.show()
            

        fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [1,1]},figsize=(10,6))
        rnl = 1 - all_regions_MI["GMI"]/all_regions_MI["TMI"]#(all_regions_MI["TMI"]-all_regions_MI["GMI"])/all_regions_MI["sGMI"]#
        num_reg_shown = np.min([10, np.count_nonzero(np.isfinite(corrected))])
        sa = np.argsort(corrected)[::-1]
        last = sa[-2:]
        if np.count_nonzero(np.isnan(corrected))>=2:
            nons = np.argsort(regions_ks_stat)[1::-1]
        else:
            nons = np.array([])
        to_show = np.roll(sa,-np.count_nonzero(np.isnan(corrected)))[:num_reg_shown]
        plt.sca(ax[0])
        plt.boxplot([rnl[i,:] for i in to_show]+[rnl[i,:] for i in last]+[rnl[i,:] for i in nons], showmeans=True)
        plt.xlabel(f"{num_reg_shown} (most) significant regions")
        plt.ylim(plt.ylim())
        plt.vlines([num_reg_shown+0.5, num_reg_shown+2.5], *plt.ylim(), colors = "green", linestyles="dashed", lw=1)
        plt.ylabel("Relative non linearity")
        if region_positions is not None:
            labels = region_positions.loc[np.concatenate([to_show,last,nons])].Labels.to_numpy()
        else:
            labels = np.arange(num_reg_shown+4)+1
        plt.xticks(np.arange(num_reg_shown+2+len(nons))+1, labels, rotation=90)
        plt.xlim(plt.xlim())
        plt.hlines([0,0.1],*plt.xlim(), ["k","r"],["solid","dashed"])

        plt.sca(ax[1])
        top_reg = np.argmax(corrected)
        top_sub = np.argsort(rnl[top_reg,:])[-15:]
        plt.barh(np.arange(15),all_regions_MI["TMI"][top_reg,top_sub],color="darkseagreen", label="Total MI")
        plt.barh(np.arange(15),all_regions_MI["GMI"][top_reg,top_sub], xerr=all_regions_MI["sGMI"][top_reg,top_sub],color="indianred", label="Surrogates MI")
        plt.ylabel(f"15 most nonlinear subjects for most significant region ({region_positions.loc[to_show[0],'Labels'] if region_positions is not None and len(to_show)>0 else '-'})")
        plt.xlabel("MI")
        plt.legend(frameon=False,loc=0)
        plt.show()

        plt.scatter(regions_ks_stat[np.isnan(corrected)], siginreg[np.isnan(corrected)], color="gray", label="non sig. KS", alpha=0.6)
        plt.scatter(regions_ks_stat[np.isfinite(corrected)], siginreg[np.isfinite(corrected)], color="blue", label="sig. KS", alpha=0.6)
        plt.xlabel("Region KS statistics")
        plt.ylabel("Sig. non-lin. pairs for region")
        plt.legend(loc=0)
        plt.title(subset_description+subset_na+f" - methods comparison - r: {pearsonr(regions_ks_stat,siginreg)[0]:.3}")
        plt.show()

        zscores = (all_regions_MI["TMI"]-all_regions_MI["GMI"])/all_regions_MI["sGMI"]
        zcorrks = np.array([pearsonr(zscores[:,i], regions_ks_stat) for i in range(subj_num)])

        thresh = HolmThresholdFromP(zcorrks)
        mask = zcorrks[:,1]<thresh
        signum = np.sum(mask)
        plt.scatter(np.arange(signum), np.sort(zcorrks[mask,0])[::-1])
        plt.scatter(np.arange(signum, subj_num), np.sort(zcorrks[np.logical_not(mask),0])[::-1], color="gray")
        plt.ylabel("Correlation z-score to ks-statistics")
        plt.xlabel("Subject")
        plt.show()
        
        # return siginreg

# fMRI


0 3 24. Frontal_Sup_Medial_R [9.1, 50.84, 30.22]

0 3 39. ParaHippocampal_L [-21.17, -15.95, -20.7]

0 2 7. Frontal_Mid_L [-33.43, 32.73, 35.46]

0 2 72. Caudate_R [14.84, 12.07, 9.42]

0 1 16. Frontal_Inf_Orb_R [41.22, 32.23, -11.91]

0 1 25. Frontal_Med_Orb_L [-5.17, 54.06, -7.4]

0 1 75. Pallidum_L [-17.75, -0.03, 0.21]

0 1 77. Thalamus_L [-10.85, -17.56, 7.98]

1 2 26. Frontal_Med_Orb_R [8.16, 51.67, -7.13]

1 2 83. Temporal_Pole_Sup_L [-39.88, 15.14, -20.18]

2 3 88. Temporal_Pole_Mid_R [44.22, 14.55, -32.23]

In [ ]:
plt.imshow(cor)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt 
from scipy.stats import pearsonr, ks_1samp, uniform
from tqdm import tqdm

def HolmThresholdFromP (p_values:np.ndarray):
    """Returns the Holm-Bonferroni threshold given an array of p-values.
    NOTA BENE: reject the null hypotheses when **strictly smaller** than this thresold. This corresponds to the *p-value* of the first non-rejected null hypothesis.

    Parameters
    ----------
    p_values : np.ndarray
        The *p-values* to consider, can be N-dimensional, will be flattened.

    Returns
    -------
    float
        The *p-value* of the first non-rejected hypothesis.
    """
    sorted_p = np.sort(p_values.flatten())
    good = sorted_p < 0.05/(sorted_p.size-np.arange(sorted_p.size))
    which = np.argmin(good)
    
    if which==0 and sorted_p[-1]<0.05:
        return 0.05
    
    return sorted_p[which]
def get_ks_maps(results_file, subset_description, subset_id, output_prefix, pair_num, subj_num, elec_num):
    region_quantiles=np.zeros((subj_num, pair_num))
    if os.path.isfile(f"../../NonLinearityResultsNew/localised/{output_prefix}_region_quantiles_{subset_id}.npy"):
        region_quantiles = np.load(f"../../NonLinearityResultsNew/localised/{output_prefix}_region_quantiles_{subset_id}.npy")
        print(f"Region quantiles for {subset_description} {subset_id} loaded")
    else:
        for s in tqdm(range(subj_num), "Subject"):
            pat = np.abs(np.load(results_file.format(subset_id, s)))
            true = pat[:,0]
            surr = np.sort(pat[:,1:], 1)
            for i in range(pair_num):
                region_quantiles[s,i] = np.searchsorted(surr[i,:], true[i], "left")
            np.save(f"../../NonLinearityResultsNew/localised/{output_prefix}_region_quantiles_{subset_id}.npy", region_quantiles)

    if os.path.isfile(f"../../NonLinearityResultsNew/localised/{output_prefix}_ks_stat_{subset_id}.npy") and os.path.isfile(f"../../NonLinearityResultsNew/localised/{output_prefix}_ks_p_{subset_id}.npy"):
        ks_stat = np.load(f"../../NonLinearityResultsNew/localised/{output_prefix}_ks_stat_{subset_id}.npy")
        ks_p = np.load(f"../../NonLinearityResultsNew/localised/{output_prefix}_ks_p_{subset_id}.npy")
        print(f"K-S stats for {subset_description} {subset_id} loaded")
    else:
        ks_stat = np.zeros(pair_num)
        ks_p = np.zeros(pair_num)
        for i in tqdm(range(pair_num), "Pair"):
            stat, pval = ks_1samp(region_quantiles[:,i], uniform.cdf,(0,100), alternative="less")#"greater" if "_cor" in results_file else 
            ks_stat[i] = stat
            ks_p[i] = pval
        np.save(f"../../NonLinearityResultsNew/localised/{output_prefix}_ks_stat_{subset_id}.npy", ks_stat)
        np.save(f"../../NonLinearityResultsNew/localised/{output_prefix}_ks_p_{subset_id}.npy", ks_p)

    threshold = HolmThresholdFromP(ks_p)
    tot = np.sum(ks_p<threshold)
    print("Threshold:", threshold, ", at stat:", ks_stat[np.isclose(ks_p,threshold)][:1], ", tot: ", tot)
    
    mappa = np.zeros([elec_num,elec_num])
    mappa[np.triu_indices(elec_num,1)]=ks_stat#corrected
    mappa+=mappa.T
    return mappa, ks_stat[np.isclose(ks_p,threshold)][0], tot
    
def get_corr_maps (temp, subset_id, subj_num, elec_num):
    corrs=[]
    for i in range(subj_num):
        corrs.append(np.load(temp.format(subset_id,i))[:]**2)#,0])
    avcor = np.mean(np.concatenate(corrs,-1).reshape([subj_num,-1]),0)
    cor = np.zeros([elec_num,elec_num])
    cor[np.triu_indices(elec_num,1)]=avcor
    cor+=cor.T
    return cor



In [ ]:
pair_num = 195*97
subj_num = 50
elec_num = 195
subset_names = [r"$\delta$",r"$\theta$",r"$\alpha$",r"$\beta_{LOW}$",r"$\beta_{MID}$",r"$\beta_{HIGH}$",r"$\gamma$",r"$\beta_{ALL}$",r"$1-40 Hz$"]
cmap = plt.cm.viridis
cmap.set_over("red")
for subset_num in range(1,10):
    upper = None
    fig, ax = plt.subplots(1,3, figsize=(12,5))
    fig2, ax2 = plt.subplots(1,2, figsize=(8,4), sharey=True)
    subset_id = f"{subset_num}"
    temp = "/home/raffaelli/NonLinearity/NonLinearityResultsNew/NEW_EEG_fixedSamples_band{}_bin10/subject{:02}_10_cor.npy"
    cor = get_corr_maps(temp, subset_id, subj_num, elec_num)
    x = cor[np.triu_indices(elec_num,1)]
    for i, (SHADOW, shap) in enumerate((["",""], ["SHADOW", "_sha"])):
        results_file = "../../NonLinearityResultsNew/NEW_EEG_fixedSamples_band{}_bin10/subject{:02}"+f"_10{shap}.npy"
        subset_description = "EEG band "
        output_prefix = f"electrodeEEG{shap}"
        maps, thresh, tot = get_ks_maps(results_file, subset_description, subset_id, output_prefix, pair_num, subj_num, elec_num)
        if upper is None:
            upper = np.max(maps)
        plt.sca(ax[i])
        plt.imshow(maps,vmax=upper, cmap=cmap)# vmax=0.41693877551020414)
        print("MI", np.max(maps))
        plt.title(f"KS-statistics {SHADOW} ({tot})")
        clb = plt.colorbar(shrink=0.5)
        clb.ax.hlines(thresh, 0,1, "orange")
        clb.ax.text(1,thresh, f"{thresh:.2}")

        plt.sca(ax2[i])
        y = maps[np.triu_indices(elec_num,1)]
        plt.scatter(x,y)
        plt.ylabel(f"KS-stat {SHADOW}")
        plt.xlabel("Correlation$^2$")
        plt.title(f"r={pearsonr(x,y)[0]:.3}")
    plt.sca(ax[2])
    plt.imshow(cor)
    plt.title("Correlation$^2$")
    plt.colorbar(shrink=0.5)
    plt.suptitle(f"band {subset_names[subset_num-1]}",y=0.8)


    fig.show()
    fig2.show()
    plt.show()

In [ ]:
subset_num, len(subset_names)

In [ ]:
pair_num = 4005
subj_num = 245
elec_num = 90

for subset_num in [400, 1000, 3000]:
    for SHADOW, shap in (["",""], ["SHADOW", "_sha"]):
        temp = "/home/raffaelli/NonLinearity/NonLinearityResultsNew/eso245_aal_raw_90_synthetic_{}_bin9/subject{:02}_9_cor.npy"
        results_file = "/home/raffaelli/NonLinearity/NonLinearityResultsNew/eso245_aal_raw_90_synthetic_{}_bin9/subject{:02}"+f"_9{shap}.npy"
        subset_description = "fMRI AAL90 "
        subset_id = f"SYNTH{subset_num}"
        output_prefix = f"fMRI_AAL90_cor_MI{shap}"
        maps = get_ks_maps(results_file, subset_description, subset_id, output_prefix, pair_num, subj_num, elec_num)

        subset_description = "fMRI AAL90 cor "
        output_prefix = "fMRI_AAL90_cor_COR"
        maps2 = get_ks_maps(temp, subset_description, subset_id, output_prefix, pair_num, subj_num, elec_num)

        cor = get_corr_maps(temp, subset_id, subj_num, elec_num)
        fig, ax = plt.subplots(1,3, figsize=(12,5))
        plt.sca(ax[0])
        plt.imshow(maps, vmax=0.41693877551020414)
        print("MI", np.max(maps))
        plt.title(f"KS-statistics {SHADOW}")
        plt.colorbar(shrink=0.5)
        plt.sca(ax[1])
        plt.imshow(maps2, vmax=0.783061224489796)#, vmin=0.4
        print("cor", np.max(maps2))
        plt.title("KS-statistics from COR")
        plt.colorbar(shrink=0.5)#extend="min"
        plt.sca(ax[2])
        plt.imshow(cor)
        plt.title("Correlation")
        plt.colorbar(shrink=0.5)
        plt.suptitle(f"synth {subset_num}")
        plt.show()
        fig, ax = plt.subplots(1,3, figsize=(15,4))
        x = cor[np.triu_indices(elec_num,1)]

        y = maps[np.triu_indices(elec_num,1)]
        plt.sca(ax[0])
        plt.scatter(x,y)
        plt.ylabel(f"KS-stat {SHADOW}")
        plt.xlabel("Correlation")
        plt.title(f"r={pearsonr(x,y)[0]:.3}")

        y = maps2[np.triu_indices(elec_num,1)]
        plt.sca(ax[1])
        plt.scatter(x,y)
        plt.ylabel("KS-stat from COR")
        plt.xlabel("Correlation")
        plt.title(f"r={pearsonr(x,y)[0]:.3}")

        x = maps[np.triu_indices(elec_num,1)]
        plt.sca(ax[2])
        plt.scatter(x,y)
        plt.ylabel("KS-stat from COR")
        plt.xlabel(f"KS-stat {SHADOW}")
        plt.title(f"r={pearsonr(x,y)[0]:.3}")

        plt.show()

In [ ]:
plt.hist(maps.flatten())

In [ ]:
CO = np.load(f"/home/raffaelli/NonLinearity/NonLinearityResultsNew/eso245_aal_raw_90_CORRNORSAM_bin9/subject01_9_cor.npy")
RQ = np.load(f"../../NonLinearityResultsNew/localised/fMRI_AAL90_cor_MI_region_quantiles_CORRNORSAM.npy")
KS = np.load(f"../../NonLinearityResultsNew/localised/fMRI_AAL90_cor_MI_ks_stat_CORRNORSAM.npy")

In [ ]:
np.argsort(KS)[-10:]
CO.shape

In [ ]:
a = np.random.randint(20,size=20)
b = 2 * a
print(a, np.sort(b)[np.argsort(np.argsort(a))])

In [ ]:
a = np.tile(np.arange(8),(6,1))
b = a[:,::-1]
np.reshape(a[np.tile(np.arange(6)[np.newaxis,:],(8,1)).T.flatten(), b.flatten()],(6,-1))==b

In [ ]:
from scipy.stats import beta, t, nct
from tqdm import tqdm
from matplotlib.colors import LogNorm

In [ ]:
gt = RQ[1,:]>50

In [ ]:
plt.hist(CO[1500,1:])
plt.vlines(CO[1500,0], 0, 10, "r")
print(CO[1500,0])

In [ ]:
from scipy.io import loadmat
import sys
sys.path.append("..")
from mienc.support import (
    a_normal_map,
    adjust_jitter,
    get_pool,
    statistics,
    surrogate,
    task_producer,
    total_mutual_information,
    pair_mutual_information,
    normalise
)
mat = loadmat("/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_aal_raw_90.mat")["subj_tcs"]

In [ ]:
print(CO.shape)
np.argmax(CO[:,0])

In [ ]:
surrogate(rs.T).shape

In [ ]:
np.corrcoef([CO[:,0], CO[np.arange(4005),np.argmax(np.abs(CO[:,1:]-CO[:,0:1]), axis=1)]-CO[:,0]])[0,1](4005)[(CO[:,0]>0.92)&gt]

In [ ]:
i1, i2 = np.array(np.triu_indices(90,1))[:,3225]
rs = np.row_stack([normalise(mat[:,i1,1]),normalise(mat[:,i2,1])])

cz = np.corrcoef(rs)[0,1]#[np.triu_indices(90,1)][1500]
mi = pair_mutual_information(rs[0],rs[1],9)
print(cz, mi)
i=0
n=np.zeros(50000)
m=np.zeros(50000)

for i in tqdm(range(50000)):
    su = surrogate(rs.T)
    ns= np.row_stack([normalise(su[:,0]),normalise(su[:,1])])
    n[i]=np.corrcoef(ns)[0,1]-cz
    m[i]=pair_mutual_information(su[:,0],su[:,1],9)-mi
    


In [ ]:
plt.hist(n, bins="auto")
plt.hist(m/10, bins="auto", alpha=0.5)
plt.show()

In [ ]:
plt.hist(n/np.min(n), bins="auto", density=True)
plt.plot(np.linspace(0,1,100), beta(0.5,0.5).pdf(np.linspace(0,1,100)))
plt.title(np.min(n))
plt.show()

In [ ]:
# plt.scatter(CO[:,0], gt)
plt.scatter(CO[:,0], CO[np.arange(4005),np.argmax(np.abs(CO[:,1:]-CO[:,0:1]), axis=1)]-CO[:,0])
plt.title(f"r={np.corrcoef([CO[:,0], CO[np.arange(4005),np.argmax(np.abs(CO[:,1:]-CO[:,0:1]), axis=1)]-CO[:,0]])[0,1]:.3}")
plt.xlabel("Correlation")
plt.ylabel("Maximum difference in surrogates")
plt.show()

In [ ]:
accu = []
for i in range(245):
    CO2 = np.load(f"/home/raffaelli/NonLinearity/NonLinearityResultsNew/eso245_aal_raw_90_CORR_bin9/subject{i:02}_9_cor.npy")
    accu.append((CO2[np.arange(4005),np.argmax(np.abs(CO2[:,1:]-CO2[:,0:1]), axis=1)]-CO2[:,0])[CO2[:,0]<1])

In [ ]:
plt.hist(np.concatenate(accu).flatten(), bins=500, density=True)
plt.plot(np.linspace(-0.02,0.02,500), nct.pdf(np.linspace(-0.02,0.02,500),1,0.5,0,0.00001))
# plt.plot(np.linspace(-0.003,0.005,500), 1e4/np.exp(np.abs(3000*np.linspace(-0.003,0.005,500))))
plt.yscale("log")
plt.show()

In [ ]:
RQ.shape

In [ ]:
accu = []
accu2 = []
for i in range(245):
    CO2 = np.load(f"/home/raffaelli/NonLinearity/NonLinearityResultsNew/eso245_aal_raw_90_CORR_bin9/subject{i:02}_9_cor.npy")
    accu.append(CO2[:,0])#np.mean(CO2[:,1:], axis=1)-CO2[:,0])#)
    MI2 = np.load(f"/home/raffaelli/NonLinearity/NonLinearityResultsNew/eso245_aal_raw_90_CORR_bin9/subject{i:02}_9.npy")
    accu2.append(np.mean(MI2[:,1:], axis=1)-MI2[:,0])
    # accu2.append(RQ[i,:])
plt.scatter(np.concatenate(accu).flatten(),np.concatenate(accu2).flatten())
plt.show()


In [ ]:
plt.hist2d(np.concatenate(accu).flatten(),np.concatenate(accu2).flatten(),20, norm=LogNorm())
plt.show()

In [ ]:
plt.hist2d(-0.5*np.log(1-cor[np.triu_indices(90,1)]**2),np.nan_to_num(maps)[np.triu_indices(90,1)],20, norm=colors.LogNorm())
plt.title("r={:.3}".format(pearsonr(-0.5*np.log(1-cor[np.triu_indices(90,1)]**2),np.nan_to_num(maps)[np.triu_indices(90,1)])[0]))
plt.xlabel("correlation")
plt.ylabel("KS-statistics")
plt.colorbar()
plt.show()

In [ ]:
-0.5*np.log(1-0.795**2)

In [ ]:
# mat=loadmat("/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_aal_raw_90.mat")["subj_tcs"].flatten()
# mat=loadmat("/home/raffaelli/NonLinearity/NonLinearityData/eso245_aal_raw_90_filtered.mat")["subj_tcs"].flatten()
mat=loadmat("/home/raffaelli/NonLinearity/NonLinearityData/EEG_el_so_BLP_NEW/NEW_EEG_fixedSamples_band7.mat")["EEG"].flatten()
import scipy.stats as stats

In [ ]:
plt.hist(mat, bins="auto", density=True)
sigma=np.std(mat)
mu=np.mean(mat)
x = np.linspace(mu - 5*sigma, mu + 5*sigma, 100)
plt.plot(x, stats.norm.pdf(x, mu, sigma))
plt.yscale("log")
plt.show()

In [ ]:
from scipy.io import savemat
simulated = np.empty([3000,90,245])
for i in tqdm(range(245)):
    CO2 = np.load(f"/home/raffaelli/NonLinearity/NonLinearityResultsNew/eso245_aal_raw_90_CORR_bin9/subject{i:02}_9_cor.npy")
    newCo = np.zeros([90,90])
    newCo[np.triu_indices(90,1)]=CO2[:,0]
    newCo+=newCo.T
    newCo[np.diag_indices(90)]=1
    simulated[:,:,i] = np.random.multivariate_normal(np.zeros(90),newCo, 3000)

savemat("/home/raffaelli/NonLinearity/NonLinearityData/eso245_aal_raw_90_synthetic.mat", {"subj_tcs":simulated})


In [ ]:
def surro_devia(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps, cut_position):
    global MAX_CM, MAX_CM2
    corrct = Corrector(nbins, nsteps, folder_name=os.path.dirname(results_file).format(subset_identifiers[0]), config="../config.ini", cache_dir="../cache/")
    corrct.compute_correction()
    all_t=[]
    all_s=[]
    all_d=[]
    for subset_id, subset_na in zip(subset_identifiers, subset_names):
        for s in tqdm(range(subj_num), "Subject"):
            pat = np.load(results_file.format(subset_id, s))#corrct.correct()
            all_t.append(pat[:,0])
            all_s.append(np.mean(pat[:,1:], 1))
            all_d.append(np.std(pat[:,1:], 1))
    real = np.concatenate(all_t)
    surro = np.concatenate(all_s)
    devia = np.concatenate(all_d)
    return real, surro, devia


In [ ]:
pair_num = 4005
subj_num = 245
elec_num = 90
quantiles = [.6,.9]
cut_pos = (-15,-75,27)
results_file = "../../NonLinearityResults/eso245_aal_{}_90_bin9/patient{:02}_9.npy"
subset_description = "fMRI AAL90 "
subset_identifiers = ["raw"]#["strin", "mod", "strin"]# 
subset_names = ["FILT"]#["strin"]# 
output_prefix = "fMRI_AAL90"
region_positions = pd.read_csv("../auxiliary_data/AAL_90regions.csv").loc[slice(None), ["Labels","x","y","z"]]
region_positions["Labels"] = region_positions["Labels"].apply(lambda x: x.split()[1])
nbins = 9
nsteps = loadmat(f"/home/raffaelli/NonLinearity/NonLinearityData/eso245_aal_raw_90_filtered.mat")["subj_tcs"].shape[0]
import matplotlib.colors as colors
real, surro, devia = surro_devia(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps,cut_pos)


In [ ]:
plt.hist2d(surro, devia/surro, 30, norm=colors.LogNorm())
plt.colorbar()
plt.xlabel("Surrogate MI")
plt.ylabel("Relative variance")
plt.plot([0,1.6],[0.005,0.01], ":k")
plt.hlines([0.25],*plt.xlim(),["r"],'solid')
plt.show()
plt.hist2d(surro, (real-surro)/surro, 30, norm=colors.LogNorm())
plt.colorbar()
plt.xlabel("Surrogate MI")
plt.ylabel("Surrogates relative difference")
plt.plot([0,1.6],[0.005,0.01], ":k")
plt.hlines([0.25],*plt.xlim(),["r"],'solid')
plt.show()

In [ ]:
def surro_95(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps, cut_position):
    global MAX_CM, MAX_CM2
    corrct = Corrector(nbins, nsteps, folder_name=os.path.dirname(results_file).format(subset_identifiers[0]), config="../config.ini", cache_dir="../cache/")
    corrct.compute_correction()
    all_t=[]
    all_s=[]
    all_d=[]
    for subset_id, subset_na in zip(subset_identifiers, subset_names):
        for s in tqdm(range(subj_num), "Subject"):
            pat = np.load(results_file.format(subset_id, s))#corrct.correct()
            all_t.append(pat[:,0])
            all_s.append(np.mean(pat[:,1:], 1))
            all_d.append(np.sort(pat[:,1:], 1)[:,-5])
    real = np.concatenate(all_t)
    surro = np.concatenate(all_s)
    devia = np.concatenate(all_d)
    return real, surro, devia


In [ ]:
pair_num = 4005
subj_num = 245
elec_num = 90
quantiles = [.6,.9]
cut_pos = (-15,-75,27)
results_file = "../../NonLinearityResults/eso245_aal_{}_90_bin9/patient{:02}_9.npy"
subset_description = "fMRI AAL90 "
subset_identifiers = ["raw"]#["strin", "mod", "strin"]# 
subset_names = ["FILT"]#["strin"]# 
output_prefix = "fMRI_AAL90"
region_positions = pd.read_csv("../auxiliary_data/AAL_90regions.csv").loc[slice(None), ["Labels","x","y","z"]]
region_positions["Labels"] = region_positions["Labels"].apply(lambda x: x.split()[1])
nbins = 9
nsteps = loadmat(f"/home/raffaelli/NonLinearity/NonLinearityData/eso245_aal_raw_90_filtered.mat")["subj_tcs"].shape[0]
import matplotlib.colors as colors
real, surro, perc95 = surro_95(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps,cut_pos)


In [ ]:
# plt.hist2d(np.log10(surro), np.log10(perc95/surro), 30, norm=colors.LogNorm())#0.875
plt.hist2d(surro, perc95/surro, 30, norm=colors.LogNorm())
plt.colorbar()
plt.xlabel("Surrogate MI")
plt.ylabel("Relative 95%ile")
plt.plot([0,1.6],[0.005,0.01], ":k")
plt.hlines([1.5],*plt.xlim(),["r"],'solid')
plt.show()
plt.hist2d(surro, real/surro, 30, norm=colors.LogNorm())
plt.colorbar()
plt.xlabel("Surrogate MI")
plt.ylabel("True relative")
plt.plot([0,1.6],[0.005,0.01], ":k")
plt.hlines([1.5],*plt.xlim(),["r"],'solid')
plt.show()

In [ ]:
cmap = plt.cm.magma  # define the colormap
# extract all colors from the .jet map
cmaplist = [base_palette(i) for i in range(base_palette.N)]

In [ ]:
len(cmaplist), base_palette.N, 256/10, 25*4+26*6, np.concatenate([np.arange(6)*25, 151+np.arange(5)*26])

In [ ]:
pair_num = 4005
subj_num = 80
elec_num = 90
quantiles = [.6,.9]
cut_pos = (-15,-75,27)
results_file = "/home/raffaelli/NonLinearity/NonLinearityResultsNew/eso245_aal_raw_90_filtered_{}_bin8/subject{:02}_8.npy"
subset_description = "fMRI AAL90 "
subset_identifiers = ["FILT"]#["strin"]# 
subset_names = ["FILT"]#["strin"]# 
output_prefix = "fMRI_AAL90"
region_positions = pd.read_csv("../auxiliary_data/AAL_90regions.csv").loc[slice(None), ["Labels","x","y","z"]]
region_positions["Labels"] = region_positions["Labels"].apply(lambda x: x.split()[1])
nbins = 9
nsteps = loadmat(f"/home/raffaelli/NonLinearity/NonLinearityData/eso245_aal_raw_90_filtered.mat")["subj_tcs"].shape[0]
ks_stat_true_p = summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps,cut_pos)

In [ ]:
pair_num = 4005
subj_num = 245
elec_num = 90
quantiles = [.6,.9]
cut_pos = (-15,-75,27)
results_file = "../../NonLinearityResults/eso245_aal_{}_90_bin9/subject{:02}_9.npy"
subset_description = "fMRI AAL90 "
subset_identifiers = ["raw", "mod", "strin"]#["strin"]# 
subset_names = ["raw", "mod", "strin"]#["strin"]# 
output_prefix = "fMRI_AAL90"
region_positions = pd.read_csv("../auxiliary_data/AAL_90regions.csv").loc[slice(None), ["Labels","x","y","z"]]
region_positions["Labels"] = region_positions["Labels"].apply(lambda x: x.split()[1])
nbins = 9
nsteps = loadmat(f"/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_aal_strin_90.mat")["subj_tcs"].shape[0]
ks_stat_true_p = summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps,cut_pos)
subset_description = "SHA fMRI AAL90 "
output_prefix = "fMRI_AAL90_sha"
results_file = "../../NonLinearityResults/eso245_aal_{}_90_bin9/subject{:02}_9_sha.npy"
sha_steps = nsteps#int(5e3//nsteps)*
ks_stat_shad_p = summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, sha_steps,cut_pos)

In [ ]:
pair_num = 4005
subj_num = 245
elec_num = 90
quantiles = [.6,.9]
cut_pos = (-15,-75,27)
results_file = "../../NonLinearityResultsNew/eso245_aal_{}_90_ExtSha_bin8/subject{:02}_8.npy"
subset_description = "fMRI AAL90 ExtSha "
subset_identifiers = ["strin"]# ["raw", "mod", "strin"]
subset_names = ["strin"]# ["raw", "mod", "strin"]
output_prefix = "fMRI_AAL90_ExtSha"
region_positions = pd.read_csv("../auxiliary_data/AAL_90regions.csv").loc[slice(None), ["Labels","x","y","z"]]
region_positions["Labels"] = region_positions["Labels"].apply(lambda x: x.split()[1])
nbins = 8
nsteps = loadmat(f"/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_aal_strin_90.mat")["subj_tcs"].shape[0]
ks_stat_true_p = summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps,cut_pos)
subset_description = "SHA fMRI AAL90 ExtSha "
output_prefix = "fMRI_AAL90_ExtSha_sha"
results_file = "../../NonLinearityResultsNew/eso245_aal_{}_90_ExtSha_bin8/subject{:02}_8_sha.npy"
sha_steps = int(5e3//nsteps)*nsteps
ks_stat_shad_p = summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, sha_steps,cut_pos)

In [ ]:
plt.hist2d(ks_stat_true_p, ks_stat_shad_p,[12,5], [[-0.5,11.5],[-0.5,4.5]])
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.vlines(0.1265306, *plt.ylim(), "k", "dotted")
plt.hlines(0.1161324, *plt.xlim(), "k", "dotted")
plt.xlabel("True")
plt.ylabel("Shadow")
plt.show()
either = (ks_stat_true_p+ks_stat_shad_p)>0
print(pearsonr(ks_stat_true_p[either], ks_stat_shad_p[either]))
pearsonr(ks_stat_true_p, ks_stat_shad_p)

In [ ]:
np.where(ks_stat_true_p==3)

In [ ]:
for i in range(90):
    if ks_stat_true_p[i]+1<ks_stat_shad_p[i]:
        print(i, ks_stat_true_p[i], ks_stat_shad_p[i], aal_atlas_centers.iloc[i]["Labels"], aal_atlas_centers.iloc[i, 2:6].tolist())

In [ ]:
print(nbins, sha_steps, os.path.dirname(results_file).format(subset_identifiers[0]))

In [ ]:
corrct = Corrector(nbins, sha_steps, folder_name=os.path.dirname(results_file).format(subset_identifiers[0]), config="../config.ini", cache_dir="../cache/", verbose=True, workers=22)
corrct.compute_correction()

In [ ]:
pair_num = 4005
subj_num = 245
elec_num = 90
quantiles = [.6,.9]
results_file = "../../NonLinearityResults/eso245_aal_{}_90_bin9/patient{:02}_9.npy"
subset_description = "fMRI AAL90 "
subset_identifiers = ["strin"]# ["raw", "mod", "strin"]
subset_names = ["strin"]# ["raw", "mod", "strin"]
output_prefix = "fMRI_AAL90"
region_positions = pd.read_csv("../auxiliary_data/AAL_90regions.csv").loc[slice(None), ["Labels","x","y","z"]]
region_positions["Labels"] = region_positions["Labels"].apply(lambda x: x.split()[1])
nbins = 9
nsteps = loadmat(f"/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_aal_strin_90.mat")["subj_tcs"].shape[0]
summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps,cut_pos)
subset_description = "SHA fMRI AAL90 "
output_prefix = "fMRI_AAL90_sha"
results_file = "../../NonLinearityResults/eso245_aal_{}_90_bin9/patient{:02}_9_sha.npy"
summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps,cut_pos)

In [ ]:
pair_num = 4005
subj_num = 245
elec_num = 90
for prep in ["raw", "mod", "strin"]:
    ks_stat = np.load(f"../../NonLinearityResultsNew/localised/fMRI_AAL90_ks_stat_{prep}.npy")
    ks_p = np.load(f"../../NonLinearityResultsNew/localised/fMRI_AAL90_ks_p_{prep}.npy")
    thresh = HolmThresholdFromP(ks_p)

    corrected = np.full(pair_num, np.nan)
    corrected[ks_p<thresh] = ks_stat[ks_p<thresh]
    mappa = np.zeros([elec_num,elec_num])
    mappa[np.triu_indices(elec_num,1)]=corrected
    mappa+=mappa.T
    siginreg = np.sum(mappa>0, 1)

    Left = siginreg[::2]
    Right = siginreg[1::2]
    print(prep, pearsonr(Left, Right))

# EEG - scalp electrode

In [ ]:
region_positions.describe()

In [ ]:
plane_distance = 10
source_z = region_positions.z.min()
diameter = region_positions.z.max()-source_z
region_positions["XP"] = region_positions.x/(region_positions.z-source_z+plane_distance)*(diameter+plane_distance)
region_positions["YP"] = region_positions.y/(region_positions.z-source_z+plane_distance)*(diameter+plane_distance)
plt.scatter(region_positions.XP, region_positions.YP, s=5)
plt.gca().set_aspect('equal')
plt.show()

In [ ]:

vor = Voronoi(np.concatenate([region_positions[["XP","YP"]],np.transpose([22*np.sin(np.linspace(0,2*np.pi,100)),22*np.cos(np.linspace(0,2*np.pi,100))-0.5])]))
voronoi_plot_2d(vor, show_vertices=False, point_size=4)

In [ ]:
from scipy.spatial import Voronoi, voronoi_plot_2d

vor = Voronoi(np.concatenate([region_positions[["XP","YP"]],np.transpose([37*np.sin(np.linspace(0,2*np.pi,100)),37*np.cos(np.linspace(0,2*np.pi,100))-5])]))
for i, v in enumerate(siginreg):
    region = vor.regions[vor.point_region[i]]
    vertices =np.array([vor.vertices[n] for n in region])
    plt.fill(vertices[:,0], vertices[:,1], color=plt.cm.magma(v/vmax))
plt.gca().set_aspect('equal')
plt.show()
voronoi_plot_2d(vor, show_vertices=False, point_size=4)
plt.gca().set_aspect('equal')
plt.plot(37*np.sin(np.linspace(0,2*np.pi,50)),37*np.cos(np.linspace(0,2*np.pi,50))-5, "-r")
plot_cap(siginreg, "WOW", region_positions, cmap=plt.cm.magma, vmin=0)
plt.show()

In [ ]:
vor.regions, vor.point_region

In [ ]:
vmin=0
vmax=np.max(siginreg)
plt.cm.magma(siginreg)#, vmin, vmax)
plt.scatter(region_positions.XP, region_positions.YP, s=7, color=plt.cm.magma(siginreg/vmax))
plt.show()

In [ ]:
for i, v in enumerate(siginreg):
    region = vor.regions[vor.point_region[i]]
    vertices =np.array([vor.vertices[n] for n in region])
    plt.fill(vertices[:,0], vertices[:,1], color=plt.cm.magma(v/vmax))
plt.show()

run scalp electrode

In [ ]:
file = zipfile.PyZipFile("../../NonLinearityData/EEG_el_so_BLP_20230714.zip")
archive = h5py.File(io.BytesIO(file.read("info.mat")))
zip_file = zipfile.PyZipFile("../../NonLinearityData/EEG_el_so_BLP_20230714.zip")
infoMat = h5py.File(io.BytesIO(zip_file.read("info.mat")))
vec= np.where(infoMat["elec_in_mask"])[1]
region_positions = pd.read_csv("../auxiliary_data/electrode_positions.csv").loc[vec, ["Labels","x","y","z"]].reset_index()

cut_pos = (-15,-75,27)
pair_num = 18915
subj_num = 50
elec_num = 195
quantiles = [.7,.95]
results_file = "../../NonLinearityResultsNew/NEW_EEG_fixedSamples_band{}_bin10/subject{:02}_10.npy"
subset_description = "EEG band "
subset_identifiers = range(1,10)
subset_names = [r"$\delta$",r"$\theta$",r"$\alpha$",r"$\beta_{LOW}$",r"$\beta_{MID}$",r"$\beta_{HIGH}$",r"$\gamma$",r"$\beta_{ALL}$",r"$1-40 Hz$"]
output_prefix = "electrodeEEG"
nbins = 10
nsteps = loadmat(f"/home/raffaelli/NonLinearity/NonLinearityData/EEG_el_so_BLP_NEW/NEW_EEG_fixedSamples_band1.mat")["EEG"].shape[0]
summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps, cut_pos)

# EEG - scalp BLP

In [ ]:
file = zipfile.PyZipFile("../../NonLinearityData/EEG_el_so_BLP_20230714.zip")
archive = h5py.File(io.BytesIO(file.read("info.mat")))
zip_file = zipfile.PyZipFile("../../NonLinearityData/EEG_el_so_BLP_20230714.zip")
infoMat = h5py.File(io.BytesIO(zip_file.read("info.mat")))
vec= np.where(infoMat["elec_in_mask"])[1]
region_positions = pd.read_csv("../auxiliary_data/electrode_positions.csv").loc[vec, ["Labels","x","y","z"]].reset_index()

pair_num = 18915
subj_num = 50
elec_num = 195
quantiles = [.7,.95]
results_file = "../../NonLinearityResultsNew/NEW_electrodeBLP_fixedTime_avg2_band{}_bin8/subject{:02}_8.npy"
subset_description = "BLP band "
subset_identifiers = range(1,10)
subset_names = [r"$\delta$",r"$\theta$",r"$\alpha$",r"$\beta_{LOW}$",r"$\beta_{MID}$",r"$\beta_{HIGH}$",r"$\gamma$",r"$\beta_{ALL}$",r"$1-40 Hz$"]
output_prefix = "electrodeBLP"
nbins = 8
nsteps = loadmat(f"/home/raffaelli/NonLinearity/NonLinearityData/EEG_el_so_BLP_NEW/NEW_electrodeBLP_fixedTime_avg2_band1.mat")["EEG"].shape[0]
summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps)

# EEG - source BLP

**Doesn't work as I didn't save ALL the values for all the pairs (1e6) for all the bands, averages, and subjects (~4 TB)**

In [ ]:
assert False
region_positions = None

pair_num = 1072380
subj_num = 50
elec_num = 1465
quantiles = [.7,.95]
results_file = "../NonLinearityResultsNew/NEW_sourceBLP_fixedTime_avg2_band{}_bin8/subject{:02}_8.npy"
subset_description = "BLP band "
subset_identifiers = range(1,10)
subset_names = [r"$\delta$",r"$\theta$",r"$\alpha$",r"$\beta_{LOW}$",r"$\beta_{MID}$",r"$\beta_{HIGH}$",r"$\gamma$",r"$\beta_{ALL}$",r"$1-40 Hz$"]
output_prefix = "sourceBLP"
nbins = 8
nsteps = loadmat(f"/home/raffaelli/NonLinearity/NonLinearityData/EEG_el_so_BLP_NEW/NEW_sourceBLP_fixedTime_avg2_band1.mat")["EEG"].shape[0]
summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps)

In [ ]:
assert False

In [ ]:
def vecmod (X):
    return np.sqrt(np.sum(X**2))
def dist (X, Y):
    return vecmod(X-Y)

In [ ]:


vertices = reg.loc[slice(None), ["X","Y","Z"]].to_numpy()
T = []
for i in range(90):
    pi = i%2
    x = 30 if pi else -30
    if dist(vertices[i], np.array([x, 0, 15])) < 20:
        continue
    for j in range(i+1, 90):
        pj = j%2
        x = 30 if pj else -30
        if dist(vertices[j], np.array([x, 0, 15])) < 20:
            continue
        for k in range(j+1, 90):
            pk = k%2
            if pi+pj+pk in {1,2}:
                continue
            x = 30 if pk else -30
            if dist(vertices[k], np.array([x, 0, 15])) < 20:
                continue
            d12 = dist(vertices[i], vertices[j])
            if d12>45:
                continue
            d13 = dist(vertices[i], vertices[k])
            if d13>45:
                continue
            d23 = dist(vertices[k], vertices[j])
            if d23>45:
                continue
            if np.max(np.diff(vertices[[i,j,k],0]))>30:
                continue
            T.append([i,j,k])
fig = plt.figure()
ax = fig.add_subplot(111,projection='3d')
ax.plot_trisurf(vertices[:,0], vertices[:,1], vertices[:,2], triangles = T, edgecolor=[[0,0,0]], linewidth=1.0, alpha=0.0, shade=False)

plt.show()

In [ ]:
def read_n_lines(f, nline,ncols = 3, dtype = float, sep = ' '):
    m = np.zeros((nline, ncols), dtype = dtype)
    for i in range(nline):
        line = f.readline()
        try:
            m[i,:] = [dtype(v)  for v in line.split(sep) ]
        except:
            print('error', i, line)
    return m


def read_mesh(filename):
    with open(filename)as f:
        nline =  int(f.readline())
        coords = read_n_lines(f, nline,ncols = 3, dtype = float, sep = ' ')
        nline =  int(f.readline())
        faces = read_n_lines(f, nline,ncols = 3, dtype = int, sep = ' ')-1
    return coords, faces

In [ ]:
vertices, faces = read_mesh("BrainMesh_ICBM152.nv")

fig = plt.figure()
ax = fig.add_subplot(111,projection='3d')
ax.plot_trisurf(subs[:,0], subs[:,1], subs[:,2], triangles = newtri, edgecolor=[[0,0,0]], linewidth=1.0, alpha=0.0, shade=False)
siginreg = np.sum(mappa>0, 1)

linear = reg.iloc[siginreg==0]
mild = reg.iloc[np.logical_and(siginreg>0, siginreg<=3)]
very = reg.iloc[siginreg>3]

ax.scatter(linear.X, linear.Y, linear.Z, marker="o", color="gray")
ax.scatter(mild.X, mild.Y, mild.Z, marker="o", color="orange")
ax.scatter(very.X, very.Y, very.Z, marker="o", color="red")

ax.set_xlabel('X Label')
ax.set_ylabel('Y Label')
ax.set_zlabel('Z Label')
plt.show()

In [ ]:
from scipy.spatial import Delaunay

points = vertices[::50,:]
tri = Delaunay(points)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,projection='3d')
ax.plot_trisurf(points[:,0], points[:,1], points[:,2], triangles = tri.simplices, edgecolor=[[0,0,0]], linewidth=1.0, alpha=0.0, shade=False)

plt.show()

In [ ]:
subs = vertices[::100,:]
npt = subs.shape[0]
newtri = []
for i in tqdm(range(npt)):
    dists=[dist(subs[i], subs[j]) for j in range(i+1,npt)]
    neigh=np.argsort(dists)[:3]+i+1
    for n1 in range(3):
        for n2 in range(n1+1, 3):
            newtri.append([i, neigh[n1], neigh[n2]])


In [ ]:
pd.read_sql("../../Downloads/pat_26402_2012-12-20.sql")

# Freiburg dataset
## Extraction

These tasks are performed better by the script `../../Downloads/cleaningScript.py`

In [ ]:
from glob import glob
for file_name in glob("../../Downloads/pat_*.sql"):
    headers = {}
    data = {}
    with open(file_name) as file:
        for line in file:
            if line.startswith("INSERT"):
                pieces = line.strip()[:-1].replace("'",'"').split()
                values_ind = pieces.index("VALUES")
                header = " ".join(pieces[3:values_ind])[1:-1]
                row = " ".join(pieces[values_ind+1:])[1:-1]
                if pieces[2] in headers:
                    assert headers[pieces[2]] == header
                    data[pieces[2]].append(row)
                else:
                    headers[pieces[2]] = header
                    data[pieces[2]] = [row]
    for tab in headers:
        with open(f"{file_name[:-4]}_{tab}.csv", "w") as file:
            print(headers[tab], file=file)
            for row in data[tab]:
                print(row, file=file)

In [ ]:
tables = {}
for tab in headers:
    tmp = []
    for file_name in glob("../../Downloads/pat_*.sql"):
        true_file_name = f"{file_name[:-4]}_{tab}.csv"
        try:
            tmp.append(pd.read_csv(true_file_name,  sep=',', quotechar='"', skipinitialspace=True, engine='python'))
        except FileNotFoundError:
            print(true_file_name, " missing.")
    tables[tab]=pd.concat(tmp).reset_index(drop=True)

In [ ]:
for tab in tables:
    display(tables[tab])

In [ ]:
for tab in tables:
    tables[tab].to_csv(f"/home/raffaelli/Downloads/{tab}.csv")

In [ ]:
tables2={}
for tab in tables:
    tables2[tab]=pd.read_csv(f"/home/raffaelli/Downloads/{tab}.csv")

## Reverse engineering

### Relevant links

1. https://www.fieldtriptoolbox.org/faq/coordsys/#details-of-the-analyze-coordinate-system
1. https://eeg.sourceforge.net/ANALYZE75.pdf
1. https://imaging.mrc-cbu.cam.ac.uk/imaging/FormatAnalyze
1. http://www.grahamwideman.com/gw/brain/analyze/formatdoc.htm
1. https://eeg.sourceforge.net/mri_orientation_notes.html
1. https://bids-specification.readthedocs.io/en/stable/appendices/coordinate-systems.html


In [ ]:
%matplotlib widget
import json
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import io, h5py, zipfile
import numpy as np
from tqdm import tqdm
import os
# import mienc.support as ms
# from mienc import Corrector, NonLinearEstimator
import pandas as pd
import seaborn as sns

In [ ]:
elec_pos = pd.read_csv("/home/raffaelli/Downloads/electrode.csv", index_col=0)
elec_pos.dropna(subset=["coord_x","coord_y","coord_z"], inplace=True)
elec_arr = pd.read_csv("/home/raffaelli/Downloads/electrode_array.csv", index_col=0).drop(columns=["name","commentary"])
elec = elec_pos.join(elec_arr,on="array")
elec["sid"] = [int(str(a)[:-6]) for a in elec.index.values]
elec["eid"] = [int(str(a)[-6:-3]) for a in elec.index.values]
elec.set_index(["sid","eid"], inplace=True)
elec.dropna(axis=1, how="all", inplace=True)
elec.sort_index()
eeg_focus = pd.read_csv("/home/raffaelli/Downloads/eeg_focus.csv", index_col=0).dropna(axis=1)
eeg_focus["sid"] = [int(str(a)[:-6]) for a in eeg_focus.index.values]
eeg_focus["eid"] = [int(str(a)[-6:-3]) for a in eeg_focus.index.values]
eeg_focus.set_index(["sid","eid"], inplace=True)

### Front-back analysis

In [ ]:
eeg_focus["coronal"]=eeg_focus["localisation"].apply(lambda x: x[0])
coronal = eeg_focus.groupby("sid").apply(lambda df: "f" if "f" in df.coronal.unique() else "o" if "o" in df.coronal.unique() else None).dropna()#
frontal_sub = coronal[coronal=="f"].index
occipital_sub = coronal[coronal=="o"].index
coronal

#### Occipital subject

In [ ]:
tmp = elec.sort_index().loc[occipital_sub]
occipital_focus = tmp[tmp["focus_rel"]=="i"].index

In [ ]:
fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [8,0.1]},figsize=(8,6))
ax1 = fig.add_subplot(121,projection='3d')
ax[0].axis("off")
ax[1].axis("off")
for etype, color in zip(['strip', 'grid'], "gb"):#, 'depth', 'surf'rk
    ind = elec[elec["type"]==etype].index
    scat = ax1.scatter(elec.loc[ind, "coord_x"], elec.loc[ind, "coord_y"], elec.loc[ind, "coord_z"], marker="o", color=color, label=etype, s=2)
scat = ax1.scatter(elec.loc[occipital_sub, "coord_x"], elec.loc[occipital_sub, "coord_y"], elec.loc[occipital_sub, "coord_z"], marker="+", color="orange", label="all_1146", s=8)
scat = ax1.scatter(elec.loc[occipital_focus, "coord_x"], elec.loc[occipital_focus, "coord_y"], elec.loc[occipital_focus, "coord_z"], marker="*", color="r", label="SOZ_1146")

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
plt.legend()
plt.show()


#### Frontal subjects

In [ ]:
tmp = elec.sort_index().loc[frontal_sub]
frontal_sub_focus = tmp[tmp["focus_rel"]=="i"].index

In [ ]:
fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [8,0.1]},figsize=(8,6))
ax1 = fig.add_subplot(121,projection='3d')
ax[0].axis("off")
ax[1].axis("off")
for etype, color in zip(['strip', 'grid'], "gb"):#, 'depth', 'surf'rk
    ind = elec[elec["type"]==etype].index
    scat = ax1.scatter(elec.loc[ind, "coord_x"], elec.loc[ind, "coord_y"], elec.loc[ind, "coord_z"], marker="o", color=color, label=etype, s=2)
scat = ax1.scatter(elec.loc[frontal_sub, "coord_x"], elec.loc[frontal_sub, "coord_y"], elec.loc[frontal_sub, "coord_z"], marker="+", color="orange", label="all_1146", s=8)
scat = ax1.scatter(elec.loc[frontal_sub_focus, "coord_x"], elec.loc[frontal_sub_focus, "coord_y"], elec.loc[frontal_sub_focus, "coord_z"], marker="*", color="r", label="SOZ_1146")

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
plt.legend()
plt.show()

### Side analysis

In [ ]:
eeg_focus["side"]=eeg_focus["localisation"].apply(lambda x: x[-1])
sided = eeg_focus.groupby("sid").apply(lambda df: df.side.unique()[0] if len(df.side.unique())==1 and df.side.unique()[0]!="b" else None).dropna()
right_sub = sided[sided=="r"].index
left_sub = sided[sided=="l"].index
print(f"Only right: {len(right_sub)}, only left: {len(left_sub)}")

#### Show right

In [ ]:
tmp = elec.sort_index().loc[right_sub]
right_sub_focus = tmp[tmp["focus_rel"]=="i"].index

In [ ]:
fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [8,0.1]},figsize=(8,6))
ax1 = fig.add_subplot(121,projection='3d')
ax[0].axis("off")
ax[1].axis("off")
# for etype, color in zip(['strip', 'grid'], "gb"):#, 'depth', 'surf'rk
#     ind = elec[elec["type"]==etype].index
#     scat = ax1.scatter(elec.loc[ind, "coord_x"], elec.loc[ind, "coord_y"], elec.loc[ind, "coord_z"], marker="o", color=color, label=etype, s=2)
scat = ax1.scatter(elec.loc[right_sub, "coord_x"], elec.loc[right_sub, "coord_y"], elec.loc[right_sub, "coord_z"], marker="+", color="orange", label="all_right", s=10)
scat = ax1.scatter(elec.loc[right_sub_focus, "coord_x"], elec.loc[right_sub_focus, "coord_y"], elec.loc[right_sub_focus, "coord_z"], marker="*", color="r", label="SOZ_right")

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
plt.legend()
plt.show()

#### Show left

In [ ]:
tmp = elec.sort_index().loc[left_sub]
left_sub_focus = tmp[tmp["focus_rel"]=="i"].index

In [ ]:
fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [8,0.1]},figsize=(8,6))
ax1 = fig.add_subplot(121,projection='3d')
ax[0].axis("off")
ax[1].axis("off")
# for etype, color in zip(['strip', 'grid'], "gb"):#, 'depth', 'surf'rk
#     ind = elec[elec["type"]==etype].index
#     scat = ax1.scatter(elec.loc[ind, "coord_x"], elec.loc[ind, "coord_y"], elec.loc[ind, "coord_z"], marker="o", color=color, label=etype, s=2)
scat = ax1.scatter(elec.loc[left_sub, "coord_x"], elec.loc[left_sub, "coord_y"], elec.loc[left_sub, "coord_z"], marker="+", color="orange", label="all_left", s=8)
scat = ax1.scatter(elec.loc[left_sub_focus, "coord_x"], elec.loc[left_sub_focus, "coord_y"], elec.loc[left_sub_focus, "coord_z"], marker="*", color="r", label="SOZ_left")

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
plt.legend()

In [ ]:
import numpy as np

mi = np.power(1.05,1/365)
(7070*mi**26)*mi**39, mi, 7070*0.08/365*7

In [ ]:
elec_pos.hist("coord_y")

In [ ]:
from nilearn import datasets, plotting
atlas_data = datasets.fetch_atlas_aal()
atlas_filename = atlas_data.maps

In [ ]:
atlas_filename

In [ ]:
plotting.plot_roi(atlas_filename, title="AAL atlas", cut_coords=(0,-45,27), cmap=plt.cm.magma)

In [ ]:
region_indices = {l:i for l,i in zip(atlas_data["labels"],atlas_data["indices"])}

In [ ]:
from nilearn import datasets, plotting, image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
aal_atlas_centers = pd.read_csv("../auxiliary_data/AAL_90regions.csv")
aal_atlas_centers_labels=aal_atlas_centers["Labels"].apply(lambda x: x.split()[1]).tolist()
atlas_data = datasets.fetch_atlas_aal()
atlas_filename = atlas_data.maps
region_indices = {l:i for l,i in zip(atlas_data["labels"],atlas_data["indices"])}
good = np.array([int(region_indices[l]) for l in region_indices if l in aal_atlas_centers_labels])
steps_norm = (good-good[0])/(good[-1]-good[0])
boundaries = np.concatenate([steps_norm[:-1]+np.diff(steps_norm)/2, [1]])

base_palette = plt.cm.magma
base_palette.set_under(base_palette.get_bad())
base_palette.set_over(base_palette.get_bad())
base_palette.set_bad(color="grey")
shad = np.load(f"fMRI_AAL90_sha_KS_nonLin.npy")
real = np.load(f"fMRI_AAL90_KS_nonLin.npy")
max_ks = np.nanmax([real, shad])
min_ks = np.nanmin([real, shad])

In [ ]:
import pandas as pd
aal_atlas_centers = pd.read_csv("../auxiliary_data/AAL_90regions.csv")
aal_atlas_centers["labels"] = aal_atlas_centers["Labels"].apply(lambda x: x.split()[1])

In [ ]:
reg_loc = []
image_data = image.get_data(atlas_filename)
for l in region_indices:
    if l in aal_atlas_centers_labels:
        reg_loc.append(np.where(image_data==int(region_indices[l])))

In [ ]:
np.int32.__dict__

In [ ]:
values = np.zeros_like(image_data, dtype="int32")
for i in range(90):
    values[reg_loc[i][0],reg_loc[i][1],reg_loc[i][2]]=real[i]*2147483647
    print(i, real[i], end="\t")

In [ ]:
values[reg_loc[i][0],reg_loc[i][1],reg_loc[i][2]], real[i]

In [ ]:
import nibabel as nib
affine=image.load_img(atlas_filename).affine

In [ ]:
np.min(values[values>0]), np.max(values[values<1000])

In [ ]:
values = np.zeros_like(image_data, dtype="int32")
for i in range(90):
    values[reg_loc[i][0],reg_loc[i][1],reg_loc[i][2]]=real[i]*2147483647
affine=image.load_img(atlas_filename).affine

img = nib.nifti1.Nifti1Image(values, affine)

plotting.plot_roi(img, title="Real", cut_coords=(-15,-75,27), cmap=base_palette, vmin=0.16306122448979588*2147483647, vmax=0.3930612244897959*2147483647, colorbar=True)
plt.show()

In [ ]:
values = np.full_like(image_data,np.nan, dtype="int32")
for i in range(90):
    values[reg_loc[i][0],reg_loc[i][1],reg_loc[i][2]]=shad[i]*2147483647
affine=image.load_img(atlas_filename).affine

img = nib.nifti1.Nifti1Image(values, affine)

plotting.plot_roi(img, title="Shadow", cut_coords=(-15,-75,27), cmap=base_palette, vmin=0.16306122448979588*2147483647, vmax=0.3930612244897959*2147483647, colorbar=True)
plt.show()

In [ ]:
img.get_fdata()

In [ ]:
aal_atlas_centers_labels=aal_atlas_centers["labels"].tolist()
good = []
bad = []
for l in region_indices:
    if l in aal_atlas_centers_labels:
        good.append(region_indices[l])
    else:
        bad.append(region_indices[l])

print(bad)
print(good)
print(len(good))

In [ ]:
palette = plt.cm.magma
palette.set_under(palette.get_bad())
palette.set_over(palette.get_bad())
palette.set_bad(color="grey")
shad = np.load(f"fMRI_AAL90_sha_KS_nonLin.npy")
real = np.load(f"fMRI_AAL90_KS_nonLin.npy")
max_ks = np.nanmax([real, shad])
min_ks = np.nanmin([real, shad])
palette(0.4)
# ax1.scatter(region_positions[np.isnan(corrected)].x, region_positions[np.isnan(corrected)].y, region_positions[np.isnan(corrected)].z, marker="o", c="grey")
# scat = ax1.scatter(region_positions.x, region_positions.y, region_positions.z, marker="o", c=regions_ks_stat, cmap=palette, vmin=np.nanmin(corrected), vmax=MAX_CM2)

In [ ]:
palette.with_extremes()

In [ ]:
steps = np.array([int(g) for g in good])
steps_norm = (steps-steps[0])/(steps[-1]-steps[0])
boundaries = np.concatenate([steps_norm[:-1]+np.diff(steps_norm)/2, [1]])
boundaries

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

def cmap_from_values (values, base_cmap, vmin=None, vmax=None, boundaries=None):
    if vmin is None:
        vmin = np.min(values)
    if vmax is None:
        vmax = np.max(values)
    if boundaries is None:
        boundaries = 1 - np.linspace(0,1,len(values),False)
    cdict = {
        'red': [],
        'green': [],
        'blue': [],
        'alpha': []
    }
    tmpdict = {
        'red': [0, 0],
        'green': [0, 0],
        'blue': [0, 0],
        'alpha': [0, 0]
    }
    for i in range(90):
        tv = (values[i]-vmin)/(vmax-vmin)
        tc = base_cmap(tv)
        for j, channel in enumerate(tmpdict):
            tmpdict[channel].append(tc[j])
            cdict[channel].append(tuple(tmpdict[channel]))
            tmpdict[channel] = [boundaries[i], tc[j]]
    for j, channel in enumerate(tmpdict):
        tmpdict[channel].append(0)
        cdict[channel].append(tuple(tmpdict[channel]))

    return LinearSegmentedColormap("real", cdict)


In [ ]:
new_palette = cmap_from_values(real, palette, min_ks, max_ks, boundaries)
plotting.plot_roi(atlas_filename, title="True", cut_coords=(-15,-75,27), cmap=new_palette, vmin=2001, vmax=8302, colorbar=True)
plt.show()
new_palette = cmap_from_values(shad, palette, min_ks, max_ks, boundaries)
plotting.plot_roi(atlas_filename, title="Shadow", cut_coords=(-15,-75,27), cmap=new_palette, vmin=2001, vmax=8302, colorbar=True)
plt.show()

In [ ]:
cdict

In [ ]:
region_indices.keys()

In [ ]:
display = plotting.plot_stat_map(
    image.index_img(atlas_filename, 4),
    colorbar=False,
    title="DMN nodes in MSDL atlas",
)

# Now add as an overlay the maps for the ACC and the left and right
# parietal nodes
cmaps = [
    plotting.cm.black_blue,
    plotting.cm.black_green,
    plotting.cm.black_pink,
]
for index, cmap in zip([5, 6, 3], cmaps):
    display.add_overlay(image.index_img(atlas_filename, index), cmap=cmap)

plotting.show()

In [ ]:
plt.imshow(values[:,70,:])
plt.show()

In [ ]:
np.unique(image.get_data(atlas_filename)).shape


In [ ]:
qwer = loadmat(f"/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_cra_strin_600.mat")["subj_tcs"]
ez = np.mean(qwer==0, axis=0)


In [ ]:
np.where(np.sum(ez,axis=0)>0)

In [ ]:
np.max(np.sum(ez[:,:187],axis=0))

In [ ]:
np.where(ez[:,12])

In [ ]:
qwer[:,63,12]

In [ ]:
from scipy.io import loadmat
import matplotlib.pyplot as plt
import numpy as np

mat={}
for data in ["aal_strin_90", "cra_strin_10", "cra_strin_100", "cra_strin_300"]:
    mat[data] = loadmat(f"/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_{data}.mat")["subj_tcs"][:,:5,0]
# mat = loadmat("/mnt/DATA/Motion_fMRI/Datasets/Weird_manual_masks/eso/100center.mat")["subj_tcs"]
print(mat[data].shape)

In [ ]:
ft = np.fft.rfft(mat["aal_strin_90"], axis=0)


In [ ]:
fs = 0.5
N = 400
plt.plot(fs/N*np.arange(100),ft[:100])
plt.xlim([0.0,0.10])
plt.vlines([0.009,0.08],-30,30,"k")
plt.show()

In [ ]:
0.24/(0.02/0.009), 0.02/(0.24/0.08), (0.02/0.009), (0.24/0.08)

In [ ]:
200*91.9, 200*91.9/840

In [ ]:
400/0.666666666666/60

In [ ]:
from scipy import signal
def show (mat):
    plt.plot(np.linspace(0.008,0.11,400),signal.zoom_fft(mat[:,:],[0.008,0.11], fs=0.5, endpoint=True, axis=0), ls="--", alpha=0.7)
    plt.vlines([0.009,0.08],-30,30,"k", label="Band-pass\nfrequencies")
    plt.xlabel("f [Hz]")
    plt.xscale("log")
    plt.ylim([-30,30])
    plt.xlim([7.8e-3,0.112])

fi, ax = plt.subplots(2,2,figsize=(12,10))
for i, data in enumerate(mat):
    plt.sca(ax[i//2,i%2])
    show(mat[data])
    plt.title(data)
plt.legend(loc=0)
plt.show()

In [ ]:
np.mean(mat["cra_strin_300"]==0, axis=0)

In [ ]:
from scipy.io import loadmat, savemat
import matplotlib.pyplot as plt
import numpy as np

mat={}
for data in ["aal_strin_90", "aal_mod_90", "aal_raw_90"]:
    mat[data] = loadmat(f"/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_{data}.mat")["subj_tcs"][:,:5,0]
# mat = loadmat("/mnt/DATA/Motion_fMRI/Datasets/Weird_manual_masks/eso/100center.mat")["subj_tcs"]
print(mat[data].shape)


In [ ]:

from scipy import signal
def show (mat):
    plt.plot(np.linspace(0.008,0.15,400),signal.zoom_fft(mat[:,:],[0.008,0.15], fs=0.5, endpoint=True, axis=0), ls="--", alpha=0.7)
    plt.vlines([0.009,0.08],-30,30,"k", label="Band-pass\nfrequencies")
    plt.xlabel("f [Hz]")
    plt.xscale("log")
    plt.ylim([-30,30])
    plt.xlim([7.8e-3,0.15])

f_sample = 0.5

sos = signal.butter(4, [0.009,0.08], 'bp', False, 'sos', f_sample)
fi, ax = plt.subplots(3,2,figsize=(12,10))
for i, data in enumerate(mat):
    plt.sca(ax[i,0])
    show(mat[data])
    plt.title(data)
    filtered = signal.sosfiltfilt(sos, mat[data], axis=0)
    plt.sca(ax[i,1])
    show(filtered)
    plt.title(data+" filt")
plt.legend(loc=0)
plt.show()

In [ ]:
raw = loadmat(f"/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_aal_raw_90.mat")["subj_tcs"]

f_sample = 0.5

sos = signal.butter(4, [0.009,0.08], 'bp', False, 'sos', f_sample)

filt = signal.sosfiltfilt(sos, raw, axis=0)

savemat("/home/raffaelli/NonLinearity/NonLinearityData/eso245_aal_raw_90_filtered.mat", {"subj_tcs":filt})

In [ ]:
400**(1/3)